[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luuuuuig/Bachelor-End-Project/blob/codex-fresh-observation-analysis-2026-10-12/Focal_Case_Selection_2026-10-07.ipynb)

# Fresh review of possible improvement activities
**Review draft · 12 October 2026 · five original observation CSVs plus dated observer clarifications**

This notebook starts again from Arno's five observation files. It asks: **which observed problems deserve a closer feasibility check before choosing a focal activity?**

The dated `Arno_Measurement` CSVs are the only data files used; Yijie's 12 October clarifications and the 1 October meeting scope are recorded explicitly below. The notebook does not read kinds-of-work labels, cause registers, enriched data, old candidate lists, scores, weights or prior conclusions. Opportunity boundaries below are fresh interpretations of the raw notes, kept beside traceable source rows for review.

Recorded burden can help choose what to investigate. It cannot establish that a proposed change is feasible, accepted, beneficial or an appropriate use of AI. The result is therefore an investigation priority, not a validated final selection.


## 1. Read the five original files

Run from the downloaded or cloned repository root, with the `docs/measurement` folder beside the notebook. All records remain in the overall observation record. A separate improvement-scope flag excludes EXC and the unsplittable OTHER/EXC interval from improvement analysis. No lost-focus exclusion is applied: that judgement is not established by these raw notes.

`source_row` counts data rows from 1, excluding the header. Together with the date, it identifies the original record. Column names are harmonised only where the five files use different headings.


### Open and run this notebook

**On your computer:** run from the downloaded repository, with the five `Arno_Measurement_*.csv` files in `docs/measurement`. You can also put those five CSVs beside the notebook.

**In Colab:** click **Open in Colab** above, then **Runtime → Run all**. If the data files are missing, the setup cell asks you to upload the five original CSVs together. Download them from the [recorded source folder](https://github.com/Luuuuuig/Bachelor-End-Project/tree/212ac1cb877072dd65af6e6bca81c62441f5552c/docs/measurement), or download and extract the repository ZIP and select those five files. Uploading is needed again after a runtime reset.

For a private GitHub repository, use [Colab's GitHub chooser](https://colab.research.google.com/github), enable **Include Private Repos**, and authorize your GitHub account if prompted. You can alternatively download this `.ipynb` and use **File → Upload notebook**.

This analysis reads the five observation CSVs and applies the dated observer clarifications and meeting scope shown below. The meeting is a scope source, not an additional measurement input. Their recorded source snapshot is commit `212ac1cb877072dd65af6e6bca81c62441f5552c`; replacing local/uploaded files changes the analysis inputs. The upload step does not download data from other repository files.


In [1]:
from pathlib import Path
import sys

dates = ["2026-08-31", "2026-09-01", "2026-09-08", "2026-09-18", "2026-09-23"]
data_folder = Path("docs/measurement")
if not data_folder.is_dir():
    data_folder = Path(".")

expected_files = []
missing_files = []
for date in dates:
    filename = f"Arno_Measurement_{date}.csv"
    expected_files.append(filename)
    if not (data_folder / filename).is_file():
        missing_files.append(filename)

if missing_files and "google.colab" in sys.modules:
    from google.colab import files
    print("Upload the missing observation CSVs:", missing_files)
    uploaded = files.upload()
    for filename in expected_files:
        if filename in uploaded:
            (data_folder / filename).write_bytes(uploaded[filename])

for filename in expected_files:
    if not (data_folder / filename).is_file():
        raise FileNotFoundError(f"Missing {filename}. Put the five CSVs in {data_folder} and rerun.")


In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

dates = ['2026-08-31', '2026-09-01', '2026-09-08', '2026-09-18', '2026-09-23']
frames = []
for date in dates:
    filename = 'Arno_Measurement_' + date + '.csv'
    frame = pd.read_csv(data_folder / filename, encoding='utf-8-sig', keep_default_na=False)
    frame = frame.rename(columns={
        'Activity as written': 'Activity',
        'Result / short note': 'Note'
    })
    frame['date'] = date
    frame['source_row'] = frame.index + 1
    frames.append(frame)
observations = pd.concat(frames, ignore_index=True)

### Duration and basic checks

Duration is end time minus start time. Missing clock times remain missing; an untimed record is not a zero-minute activity. The total is a sum of recorded intervals, not a measure of uninterrupted observation time or a full working week.

In [3]:
start = pd.to_datetime(observations['Start'], format='%H:%M', errors='coerce')
end = pd.to_datetime(observations['End'], format='%H:%M', errors='coerce')
observations['minutes'] = (end - start).dt.total_seconds() / 60
observations['timed'] = observations['minutes'].notna()
observations['untimed'] = observations['minutes'].isna()
observations['has_note'] = ~observations['Note'].str.strip().isin(['', '/', '—'])
observations['case_day'] = observations['date'] + ' ' + observations['Case']

assert not observations.duplicated(['date', 'source_row']).any()
assert start.notna().equals(end.notna())
assert observations.loc[observations['timed'], 'minutes'].gt(0).all()

print('Records:', len(observations))
print('Timed records:', observations['timed'].sum())
print('Untimed records:', observations['untimed'].sum())
print('Recorded minutes:', observations['minutes'].sum(min_count=1))

Records: 287
Timed records: 224
Untimed records: 63
Recorded minutes: 751.0


### Observer clarifications: 12 October 2026

Yijie clarified that a **CHECK record with no note is a standard post-order check**. This rule applies to CHECK only. The four matching source records are 31 August rows 13 and 46, 18 September row 6, and 23 September row 46. The first three already have Monitoring as their stage; the fourth now receives it. This does not identify the field checked or the number of completed lines.

For **23 September OBS-15**, Yijie explained that the walk to Emiel was to obtain more information about the PO, followed by ordinary PO clarification in row 48. Row 47 already has Ordering; row 48 is assigned Ordering from this clarified context. The original row 47 note says information was passed on. Both that original wording and the observer's clarified purpose remain visible.

For **31 August rows 27 (OBS-04) and 57 (OBS-17), and 1 September row 14 (OBS08)**, Yijie clarified that “aftercare” meant following up on outstanding orders or deliveries. These three two-minute records are assigned **Monitoring**. The clarification applies only to these identified records. They remain EXC records and outside improvement scope.

`Van Weele stage`, `Activity` and `Note` retain the original CSV values. The analytical `stage` column applies these clarifications; `observer_clarification` records their source. No source duration changes. The separate scope rule below controls eligibility; a Monitoring stage does not make an EXC record eligible.


In [4]:
observations["stage"] = observations["Van Weele stage"].replace({
    "": "Unassigned", "/": "Unassigned", "—": "Unassigned"})
observations["observer_clarification"] = ""

blank_check = observations["Activity"].str.upper().eq("CHECK")
blank_check = blank_check & observations["Note"].str.strip().isin(["", "/", "—"])
observations.loc[blank_check, "stage"] = "Monitoring"
observations.loc[blank_check, "observer_clarification"] = (
    "Standard post-order check; clarified by Yijie on 12 October 2026.")

case_date = pd.to_datetime(observations["date"]).eq(pd.Timestamp("2026-09-23"))
obs15 = case_date & observations["Case"].eq("OBS-15")
walk = obs15 & observations["source_row"].eq(47)
clarification = obs15 & observations["source_row"].eq(48)
observations.loc[clarification, "stage"] = "Ordering"
observations.loc[walk, "observer_clarification"] = (
    "Walk to Emiel to obtain PO information; clarified by Yijie on 12 October 2026.")
observations.loc[clarification, "observer_clarification"] = (
    "Ordinary PO clarification after the walk to Emiel; clarified by Yijie on 12 October 2026.")

aftercare = pd.to_datetime(observations["date"]).eq(pd.Timestamp("2026-08-31"))
aftercare = aftercare & observations["source_row"].isin([27, 57])
aftercare = aftercare | (
    observations["date"].eq("2026-09-01") & observations["source_row"].eq(14)
)
observations.loc[aftercare, "stage"] = "Monitoring"
observations.loc[aftercare, "observer_clarification"] = (
    "Following up on outstanding orders/deliveries; clarified by Yijie on 12 October 2026.")

review_columns = ["date", "source_row", "Activity", "minutes", "Van Weele stage", "stage"]
reviewed = observations["observer_clarification"].ne("")
display(observations.loc[reviewed, review_columns])
with pd.option_context("display.max_colwidth", None):
    display(observations.loc[reviewed, ["date", "source_row", "Note", "observer_clarification"]])


,date,source_row,Activity,minutes,Van Weele stage,stage
12,2026-08-31,13,CHECK,3.0,Monitoring,Monitoring
26,2026-08-31,27,EXC,2.0,,Monitoring
45,2026-08-31,46,CHECK,1.0,Monitoring,Monitoring
56,2026-08-31,57,EXC,2.0,,Monitoring
76,2026-09-01,14,EXC,2.0,,Monitoring
160,2026-09-18,6,CHECK,5.0,Monitoring,Monitoring
274,2026-09-23,46,CHECK,2.0,,Monitoring
275,2026-09-23,47,OTHER,2.0,Ordering,Ordering
276,2026-09-23,48,CLAR,7.0,,Ordering


,date,source_row,Note,observer_clarification
12,2026-08-31,13,,Standard post-order check; clarified by Yijie on 12 October 2026.
26,2026-08-31,27,aftercare,Following up on outstanding orders/deliveries; clarified by Yijie on 12 October 2026.
45,2026-08-31,46,,Standard post-order check; clarified by Yijie on 12 October 2026.
56,2026-08-31,57,Aftercare of a PO from last week,Following up on outstanding orders/deliveries; clarified by Yijie on 12 October 2026.
76,2026-09-01,14,Exception/aftercare episode; subtype not captured reliably.,Following up on outstanding orders/deliveries; clarified by Yijie on 12 October 2026.
160,2026-09-18,6,,Standard post-order check; clarified by Yijie on 12 October 2026.
274,2026-09-23,46,,Standard post-order check; clarified by Yijie on 12 October 2026.
275,2026-09-23,47,Walk to Emiel's desk to pass on information.,Walk to Emiel to obtain PO information; clarified by Yijie on 12 October 2026.
276,2026-09-23,48,,Ordinary PO clarification after the walk to Emiel; clarified by Yijie on 12 October 2026.


### Reporting labels and improvement scope

The [1 October academic-supervisor meeting](https://github.com/Luuuuuig/Bachelor-End-Project/blob/212ac1cb877072dd65af6e6bca81c62441f5552c/docs/meetings/Academic_Supervisor_Meeting_Notes_2026-10-01.md) retains aftercare under **OTHER** in the overall record, outside the AI-improvement scope; searching physically for a received item is explicitly outside scope. Yijie's 12 October correction applies that boundary here by excluding **EXC** from improvement consideration.

`reporting_activity` maps raw `EXC` and `OTHER/EXC` to `OTHER`; `Activity` stays unchanged for traceability. `improvement_scope` separately excludes both codes. The single three-minute `OTHER/EXC` interval mixes supplier, finance and delivery-date work and cannot be split reliably, so the whole interval is excluded. **Ordinary OTHER remains eligible.**

The main comparison uses eligible rows and eligible candidate boundaries. A related CLAR, SEND, REQ or OTHER record does not rescue an excluded aftercare or physical-location target. O5 and O6 are removed as candidates; their former source memberships and every excluded raw row are retained in Appendix B.


In [5]:
observations['reporting_activity'] = observations['Activity'].replace({
    'EXC': 'OTHER', 'OTHER/EXC': 'OTHER'
})
observations['improvement_scope'] = ~observations['Activity'].isin(['EXC', 'OTHER/EXC'])
observations['scope_exclusion_reason'] = ''
observations.loc[observations['Activity'].eq('EXC'), 'scope_exclusion_reason'] = (
    'EXC outside improvement scope; retained in the overall record under reporting OTHER.'
)
observations.loc[observations['Activity'].eq('OTHER/EXC'), 'scope_exclusion_reason'] = (
    'Mixed OTHER/EXC interval cannot be split; all three minutes excluded from improvement scope.'
)
scoped_observations = observations.loc[observations['improvement_scope']].copy()
excluded_rows = observations.loc[~observations['improvement_scope']].copy()

scope_counts = []
for label, rows in [('Overall record', observations),
                    ('Improvement scope', scoped_observations),
                    ('Excluded from improvement', excluded_rows)]:
    scope_counts.append({
        'scope': label, 'records': len(rows),
        'timed_records': int(rows['timed'].sum()),
        'untimed_records': int(rows['untimed'].sum()),
        'recorded_minutes': rows['minutes'].sum(min_count=1)
    })
workload_scope_summary = pd.DataFrame(scope_counts).set_index('scope')
assert tuple(workload_scope_summary.loc['Overall record']) == (287, 224, 63, 751)
assert tuple(workload_scope_summary.loc['Improvement scope']) == (246, 184, 62, 569)
assert tuple(workload_scope_summary.loc['Excluded from improvement']) == (41, 40, 1, 182)
assert observations.loc[observations['Activity'].eq('OTHER'), 'improvement_scope'].all()
display(workload_scope_summary)

,records,timed_records,untimed_records,recorded_minutes
scope,,,,
Overall record,287,224,63,751.0
Improvement scope,246,184,62,569.0
Excluded from improvement,41,40,1,182.0


In [6]:
stage_context = observations.groupby('stage').agg(records=('source_row', 'size'))
stage_context['recorded_minutes'] = observations.groupby('stage')['minutes'].sum(min_count=1)
stage_context['eligible_records'] = scoped_observations.groupby('stage').size()
stage_context['eligible_minutes'] = scoped_observations.groupby('stage')['minutes'].sum(min_count=1)
assert stage_context.loc['Monitoring', 'recorded_minutes'] == 120
assert tuple(stage_context.loc['Unassigned', ['records', 'recorded_minutes']]) == (30, 83)
assert tuple(stage_context.loc['Unassigned', ['eligible_records', 'eligible_minutes']]) == (20, 41)
display(stage_context)


,records,recorded_minutes,eligible_records,eligible_minutes
stage,,,,
Follow-up and evaluation,22,69.0,10,21.0
Monitoring,44,120.0,29,79.0
Not applicable,12,71.0,9,26.0
Ordering,164,347.0,163,341.0
Specification,12,47.0,12,47.0
Supplier selection,3,14.0,3,14.0
Unassigned,30,83.0,20,41.0


In [7]:
daily = observations.groupby('date').agg(
    records=('source_row', 'size'), timed_records=('timed', 'sum'),
    untimed_records=('untimed', 'sum'), records_with_notes=('has_note', 'sum')
)
daily['recorded_minutes'] = observations.groupby('date')['minutes'].sum(min_count=1)
daily['eligible_records'] = scoped_observations.groupby('date').size()
daily['eligible_minutes'] = scoped_observations.groupby('date')['minutes'].sum(min_count=1)
daily['share_of_eligible_minutes_pct'] = daily['eligible_minutes'] / scoped_observations['minutes'].sum() * 100
display(daily.round(1))


,records,timed_records,untimed_records,records_with_notes,recorded_minutes,eligible_records,eligible_minutes,share_of_eligible_minutes_pct
date,,,,,,,,
2026-08-31,63,39,24,41,103.0,60,99.0,17.4
2026-09-01,50,42,8,50,161.0,40,123.0,21.6
2026-09-08,42,38,4,31,106.0,36,79.0,13.9
2026-09-18,74,57,17,51,196.0,61,149.0,26.2
2026-09-23,58,48,10,32,185.0,49,119.0,20.9


In [8]:
timed_intervals = observations.loc[observations['timed'], ['date', 'source_row', 'Case', 'Start', 'End']].copy()
timed_intervals['start_time'] = start.loc[timed_intervals.index]
timed_intervals['end_time'] = end.loc[timed_intervals.index]
timed_intervals = timed_intervals.sort_values(['date', 'start_time', 'source_row'])
timed_intervals['next_start'] = timed_intervals.groupby('date')['start_time'].shift(-1)
timed_intervals['next_source_row'] = timed_intervals.groupby('date')['source_row'].shift(-1)
overlap_mask = timed_intervals['end_time'] > timed_intervals['next_start']
overlap_rows = timed_intervals.loc[overlap_mask].copy()
overlap_rows['overlap_minutes'] = (overlap_rows['end_time'] - overlap_rows['next_start']).dt.total_seconds() / 60
display(overlap_rows[['date', 'source_row', 'Case', 'Start', 'End', 'next_source_row', 'overlap_minutes']])
# Recompute adjacency after scope filtering; the raw overlap does not carry into scope.
scoped_intervals = timed_intervals.merge(
    scoped_observations[['date', 'source_row']], on=['date', 'source_row'],
    how='inner', validate='one_to_one'
).sort_values(['date', 'start_time', 'source_row'])
scoped_intervals['next_start'] = scoped_intervals.groupby('date')['start_time'].shift(-1)
assert not scoped_intervals['end_time'].gt(scoped_intervals['next_start']).any()
print('Overlapping eligible intervals: 0')


,date,source_row,Case,Start,End,next_source_row,overlap_minutes
245,2026-09-23,17,OBS-07,11:41,11:43,18.0,1.0


Overlapping eligible intervals: 0


The raw times contain a one-minute overlap on 23 September. Both source records stay in the overall record, so **751 minutes are summed interval minutes**, including that overlap. The overlapping EXC interval is excluded from improvement scope; no overlaps remain among the eligible intervals. Neither total describes a complete working week.


**What this establishes.** The overall record contains **287 records: 224 timed and 63 untimed**, summing to **751 recorded minutes**. The improvement analysis contains **246 records: 184 timed and 62 untimed**, summing to **569 minutes**. The excluded 41 records contain 182 minutes. Eligible session totals range from 79 to 149 minutes. Comparisons therefore need both associated burden and day representation; five dates do not establish normal weekly frequency or annual savings.

With the observer clarifications, the broad Monitoring stage contains **120 minutes** and the overall unassigned group contains **30 records / 83 minutes**. Within improvement scope, **20 records / 41 minutes** remain unassigned. Stage describes process position; it is separate from the improvement-scope decision.


## 2. Broad workload context before defining opportunities

The original activity labels remain visible as source evidence. For overall reporting, EXC and OTHER/EXC appear under OTHER; this aggregation does not make them eligible. The scope column in the second table distinguishes ordinary OTHER from excluded aftercare. Eligible notes then support the more specific opportunities below.


In [9]:
activity = observations.groupby('Activity').agg(
    records=('source_row', 'size'),
    timed_records=('timed', 'sum'),
    untimed_records=('untimed', 'sum'),
    days_seen=('date', 'nunique')
)
activity['recorded_minutes'] = observations.groupby('Activity')['minutes'].sum(min_count=1)
activity = activity.sort_values('recorded_minutes', ascending=False)
display(activity)
reporting_activity = observations.groupby('reporting_activity').agg(records=('source_row', 'size'))
reporting_activity['recorded_minutes'] = observations.groupby('reporting_activity')['minutes'].sum(min_count=1)
reporting_activity['eligible_records'] = scoped_observations.groupby('reporting_activity').size()
reporting_activity['eligible_minutes'] = scoped_observations.groupby('reporting_activity')['minutes'].sum(min_count=1)
reporting_activity['excluded_minutes'] = observations.loc[~observations['improvement_scope']].groupby('reporting_activity')['minutes'].sum(min_count=1)
reporting_activity['excluded_minutes'] = reporting_activity['excluded_minutes'].fillna(0)
display(reporting_activity.sort_values('recorded_minutes', ascending=False))


,records,timed_records,untimed_records,days_seen,recorded_minutes
Activity,,,,,
EXC,40,39,1,5,179.0
PO,65,64,1,5,174.0
CLAR,42,41,1,5,155.0
CHECK,21,21,0,5,91.0
OTHER,22,21,1,5,77.0
SEND,43,36,7,5,71.0
OTHER/EXC,1,1,0,1,3.0
[uncertain],1,1,0,1,1.0
DEC,3,0,3,1,NaN


,records,recorded_minutes,eligible_records,eligible_minutes,excluded_minutes
reporting_activity,,,,,
OTHER,63,259.0,22,77.0,182.0
PO,65,174.0,65,174.0,0.0
CLAR,42,155.0,42,155.0,0.0
CHECK,21,91.0,21,91.0,0.0
SEND,43,71.0,43,71.0,0.0
[uncertain],1,1.0,1,1.0,0.0
DEC,3,NaN,3,NaN,0.0
REQ,9,NaN,9,NaN,0.0
REQ-Exact,8,NaN,8,NaN,0.0


**Interpretation.** Raw EXC accounts for **179 minutes**, with **3 minutes** under OTHER/EXC. Both are excluded from improvement analysis and reported under OTHER in the overall record. Reporting OTHER therefore totals **259 minutes**, of which the **77 minutes originally labelled OTHER remain eligible**. PO has 174, CLAR 155, CHECK 91, SEND 71 and the uncertain label 1 minute. These totals locate recorded effort; they do not reveal which part a tool could remove.

All REQ variants are untimed. They must not acquire durations from nearby rows, and they do not by themselves establish unique new arrivals or repeated visits.


In [10]:
request_mask = observations['Activity'].str.upper().str.startswith('REQ')
request_rows = observations.loc[request_mask].copy()
exact_request_mask = request_rows['Activity'].str.upper().eq('REQ-EXACT')
print('REQ-labelled records:', len(request_rows))
print('REQ-Exact records, ignoring letter case:', exact_request_mask.sum())
print('Timed REQ-labelled records:', request_rows['timed'].sum())

REQ-labelled records: 49
REQ-Exact records, ignoring letter case: 14
Timed REQ-labelled records: 0


There are **49 REQ-labelled records**, including **14 labelled REQ-Exact** when letter case is ignored. This is a count of source records. The CSVs alone do not support interpreting the other 35 as new external arrivals, or the 14 as revisits.

## 3. Define opportunities from specific notes

The following shortlist is a review of recurring or substantial problems described in the notes. It is not a universal reclassification of the 246 eligible records. O1–O4 are the eligible candidates; O5 and O6 are retained only in the exclusion audit. Eligible routine or unexplained work remains visible in the workload context and in the coverage check below.

Each selected row has one of three evidence scopes:

- **Direct (`direct`):** its note specifically describes the opportunity. Its full duration is still only associated time, not a potential saving.
- **Mixed or uncertain (`mixed_or_uncertain`):** relevant work shares an interval with other work, or the precise subject is uncertain. Keep the whole interval; do not split its minutes.
- **Contextual (`contextual`):** a related case step or broad description supplies context, but does not establish time spent on the proposed function.

The same interval can support several opportunities. Candidate totals must therefore not be added together. A row with no duration remains untimed in every view.


In [11]:
opportunity_names = {
    'O1': 'Comparison evidence',
    'O2': 'Order consolidation context',
    'O3': 'Item/document retrieval',
    'O4': 'Article/supplier information',
}
evidence_rows = []


### O1. Compare order information and document discrepancies

**Possible support to investigate.** An investigation candidate: present agreed price/date fields and their sources side by side, flag differences for buyer review.

**Boundary.** Explicit price, delivery-date or discrepancy comparison and eligible document context. The former EXC screenshot and partial-confirmation follow-up rows are excluded from this candidate. All CHECK records are shown separately as broad context; a CHECK code does not establish the checked field or suitability for this function.

**What remains unknown.** Some formats are named in the notes, but the actual source documents and approved reference values have not been inspected. Twenty-minute pre-check names a mismatch but not its field; do not silently call it price-checking time. A six-minute episode combines delivery-date adjustment with a blocked price check. All interval minutes are observed associated work, not time an aid would save.


In [12]:
# Date, source row, evidence scope, and reason for including the record.
candidate = 'O1'
evidence_rows += [
    (candidate, '2026-08-31', 45,
     'direct', 'The note identifies a missing delivery date.'),
    (candidate, '2026-09-08', 21,
     'direct', 'The note explicitly describes price comparison using quotation and website.'),
    (candidate, '2026-09-18', 36,
     'direct', 'The note explicitly names a post-order price check.'),
    (candidate, '2026-09-18', 37,
     'direct', 'The note explicitly names a post-order price check.'),
    (candidate, '2026-09-18', 38,
     'direct', 'The note explicitly names a post-order price check.'),
    (candidate, '2026-09-18', 73,
     'direct', 'The note identifies a large price difference; separate case note is unavailable here.'),
    (candidate, '2026-09-18', 74,
     'direct', 'EXC episode explicitly saves screenshots of the price difference.'),
    (candidate, '2026-09-01', 30,
     'mixed_or_uncertain', 'Delivery-time adjustment and a blocked price check share one interval; no split is supported.'),
    (candidate, '2026-09-18', 66,
     'mixed_or_uncertain', 'Pre-check and difficult mismatch are explicit; the compared field is not.'),
    (candidate, '2026-09-18', 3,
     'contextual', 'Information request about a price mismatch, not an explicitly timed comparison.'),
    (candidate, '2026-09-18', 7,
     'contextual', 'Attachment/filing steps are related document handling, not measured comparison time.'),
    (candidate, '2026-09-18', 23,
     'contextual', 'Attachment/filing steps are related document handling, not measured comparison time.'),
    (candidate, '2026-09-01', 32,
     'contextual', 'A partial supplier confirmation triggers follow-up; detecting and resolving it are not separately timed.'),
]

### O2. Make related open orders and demand visible during consolidation

**Possible support to investigate.** An investigation candidate: show relevant existing orders and demand while the buyer combines order lines; the buyer retains order and hold decisions.

**Boundary.** Explicit MAX/maximalisatie attempts and explicit additions/combination of order lines. Line additions show a related workflow, but do not by themselves prove a same-supplier search or a MAX decision.

**What remains unknown.** No observed interval isolates search, data entry and order/hold judgement. MAXOBS=? explicitly means the check was not captured; it is not evidence of zero MAX work. Same-supplier identity, required demand states and data availability are not generally established by these CSVs. Do not treat every PO minute as consolidation or use added-line outcomes as validated references for a good ordering decision.

In [13]:
# Source-row evidence for O2. These rows form one continuous list.
candidate = 'O2'
evidence_rows += [
    (candidate, '2026-08-31', 37,
     'direct', 'Untimed DEC notes explicitly identify MAX; occurrence only.'),
    (candidate, '2026-08-31', 51,
     'direct', 'Untimed DEC notes explicitly identify MAX; occurrence only.'),
    (candidate, '2026-08-31', 38,
     'mixed_or_uncertain', 'MAX and PO work share the recorded interval; the note explicitly rejects a pure decision-time interpretation.'),
    (candidate, '2026-08-31', 52,
     'mixed_or_uncertain', 'MAX and PO work share the recorded interval; the note explicitly rejects a pure decision-time interpretation.'),
    (candidate, '2026-09-01', 46,
     'mixed_or_uncertain', 'MAX is explicitly observed inside a PO episode; search, addition and outcome are not separately timed.'),
    (candidate, '2026-09-01', 47,
     'mixed_or_uncertain', 'MAX is explicitly observed inside a PO episode; search, addition and outcome are not separately timed.'),
    (candidate, '2026-09-01', 48,
     'mixed_or_uncertain', 'MAX is explicitly observed inside a PO episode; search, addition and outcome are not separately timed.'),
    (candidate, '2026-09-01', 49,
     'mixed_or_uncertain', 'MAX is explicitly observed inside a PO episode; search, addition and outcome are not separately timed.'),
    (candidate, '2026-09-01', 50,
     'mixed_or_uncertain', 'MAX is explicitly observed inside a PO episode; search, addition and outcome are not separately timed.'),
    (candidate, '2026-09-18', 58,
     'mixed_or_uncertain', 'Maximisatie or its unsuccessful attempt is explicitly named within a PO interval.'),
    (candidate, '2026-09-18', 61,
     'mixed_or_uncertain', 'Maximisatie or its unsuccessful attempt is explicitly named within a PO interval.'),
    (candidate, '2026-09-18', 64,
     'mixed_or_uncertain', 'Maximisatie or its unsuccessful attempt is explicitly named within a PO interval.'),
    (candidate, '2026-09-23', 57,
     'mixed_or_uncertain', 'Maximalisatie of an existing named-supplier PO is explicit; internal timing is not separated.'),
    (candidate, '2026-08-31', 8,
     'contextual', 'Request to add an article to an old PO and its added line are explicit; a MAX/same-supplier search is not recorded.'),
    (candidate, '2026-08-31', 10,
     'contextual', 'Request to add an article to an old PO and its added line are explicit; a MAX/same-supplier search is not recorded.'),
    (candidate, '2026-09-08', 30,
     'contextual', 'Combining or adding lines is explicit; the MAX/same-supplier search itself is not recorded.'),
    (candidate, '2026-09-08', 31,
     'contextual', 'Combining or adding lines is explicit; the MAX/same-supplier search itself is not recorded.'),
]

In [14]:
# Source-row evidence for O2. These rows form one continuous list.
candidate = 'O2'
evidence_rows += [
    (candidate, '2026-09-08', 37,
     'contextual', 'Combining or adding lines is explicit; the MAX/same-supplier search itself is not recorded.'),
    (candidate, '2026-09-08', 40,
     'contextual', 'Combining or adding lines is explicit; the MAX/same-supplier search itself is not recorded.'),
    (candidate, '2026-09-08', 41,
     'contextual', 'Combining or adding lines is explicit; the MAX/same-supplier search itself is not recorded.'),
    (candidate, '2026-09-08', 42,
     'contextual', 'Combining or adding lines is explicit; the MAX/same-supplier search itself is not recorded.'),
    (candidate, '2026-09-18', 25,
     'contextual', 'The note adds a dropped item while naming another supplier; this is not proof of same-supplier consolidation and requires clarification.'),
    (candidate, '2026-09-18', 30,
     'contextual', 'Added lines are explicit; no isolated consolidation/search time is measured.'),
    (candidate, '2026-09-18', 56,
     'contextual', 'Added lines are explicit; no isolated consolidation/search time is measured.'),
    (candidate, '2026-09-23', 2,
     'contextual', 'Lines are added to orders; the source does not identify the search or decision mechanism.'),
    (candidate, '2026-09-23', 3,
     'contextual', 'Lines are added to orders; the source does not identify the search or decision mechanism.'),
    (candidate, '2026-09-23', 4,
     'contextual', 'Lines are added to orders; the source does not identify the search or decision mechanism.'),
    (candidate, '2026-09-23', 5,
     'contextual', 'Lines are added to orders; the source does not identify the search or decision mechanism.'),
    (candidate, '2026-09-23', 8,
     'contextual', 'Lines are added to orders; the source does not identify the search or decision mechanism.'),
    (candidate, '2026-09-23', 18,
     'contextual', 'Lines are added to orders; the source does not identify the search or decision mechanism.'),
    (candidate, '2026-09-23', 39,
     'contextual', 'Lines are added to orders; the source does not identify the search or decision mechanism.'),
    (candidate, '2026-08-31', 39,
     'contextual', 'Untimed return after HOLD intention; not a measured additional consolidation interval.'),
    (candidate, '2026-09-01', 45,
     'contextual', 'HOLD is observed while MAX is explicitly unknown.'),
]

### O3. Retrieve previous item records and technical documents

**Possible support to investigate.** An investigation candidate: retrieve plausible previous order/item records and drawings with their source and revision for buyer review.

**Boundary.** Explicit historical lookup, item-history search and drawing search. Drawing preparation, completeness checking and receipt are separate associated steps; do not assert a validated completeness decision from retrieval evidence.

**What remains unknown.** Relevant documents, versions, access and acceptable reference answers are unverified. Several intervals combine search, clarification or another case; do not split their minutes. A tube-history lookup does not explicitly establish that the retrieved objects were old POs. The notes do not validate automatic determination that a drawing set is complete.

In [15]:
# Date, source row, evidence scope, and reason for including the record.
candidate = 'O3'
evidence_rows += [
    (candidate, '2026-09-01', 20,
     'mixed_or_uncertain', 'Old-PO lookup, matching alternatives and missing required information share one interval.'),
    (candidate, '2026-09-18', 28,
     'mixed_or_uncertain', 'Drawing search and asking the engineer share one interval.'),
    (candidate, '2026-09-18', 31,
     'mixed_or_uncertain', 'Colleague enquiries for OBS-11 and an OBS-10 drawing discussion share four minutes across two cases; leave unallocated.'),
    (candidate, '2026-09-23', 34,
     'mixed_or_uncertain', 'Unsuccessful drawing search and asking a colleague share one interval.'),
    (candidate, '2026-09-23', 38,
     'direct', 'Item history is explicitly searched; object type beyond tube history is unspecified.'),
    (candidate, '2026-09-18', 27,
     'contextual', 'Preparing drawings and recording missing ones is associated document work, not explicitly retrieval alone.'),
    (candidate, '2026-09-18', 40,
     'contextual', 'Drawing receipt is not measured search time.'),
    (candidate, '2026-09-18', 41,
     'contextual', 'Drawing completeness checking is a distinct judgement.'),
    (candidate, '2026-09-23', 40,
     'contextual', 'Requester confirmation follows the item-history investigation; separate referenced case note unavailable.'),
]

### O4. Prepare article and supplier information for Exact

**Possible support to investigate.** An investigation candidate: collect required article/supplier fields and supporting sources, flag missing fields, and prepare a draft for authorised review.

**Boundary.** Explicit article-entry requests, article data entry/attachments and supplier-add/update work. Same-case clarification and forwarding remain context where their actual fields are not recorded.

**What remains unknown.** Required fields and valid values are not defined in these observations. Input, source attachment, clarification and supplier selection are not interchangeable tasks. The phrase Update it in Exact does not identify whether article master data, an order or another object changed. Blank-note clarification after an item request does not establish seven minutes of extraction or entry.

In [16]:
# Date, source row, evidence scope, and reason for including the record.
candidate = 'O4'
evidence_rows += [
    (candidate, '2026-08-31', 19,
     'direct', 'Untimed request explicitly asks for article input in Exact.'),
    (candidate, '2026-08-31', 21,
     'direct', 'Input to Exact follows the explicit article-input request in the same case.'),
    (candidate, '2026-08-31', 23,
     'direct', 'Supporting information is attached to an article.'),
    (candidate, '2026-08-31', 24,
     'direct', 'Supporting information is attached to an article.'),
    (candidate, '2026-08-31', 25,
     'mixed_or_uncertain', 'Second item needs input; asking a question is the observed action.'),
    (candidate, '2026-09-01', 26,
     'direct', 'Untimed request explicitly asks to add two new items to Exact.'),
    (candidate, '2026-09-01', 33,
     'direct', 'Continuation explicitly names article-master work previously blocked by missing required data.'),
    (candidate, '2026-09-08', 26,
     'mixed_or_uncertain', 'Finding a supplier, asking another person to add it and updating Exact share four minutes.'),
    (candidate, '2026-09-23', 53,
     'direct', 'Untimed request explicitly asks to create an item.'),
    (candidate, '2026-08-31', 20,
     'contextual', 'Same article-input case contains information preparation and clarification, but fields and action split are not recorded.'),
    (candidate, '2026-08-31', 26,
     'contextual', 'Same article-input case contains information preparation and clarification, but fields and action split are not recorded.'),
    (candidate, '2026-08-31', 63,
     'contextual', 'Same article-input case contains information preparation and clarification, but fields and action split are not recorded.'),
    (candidate, '2026-08-31', 22,
     'contextual', 'Untimed price question in the article-input case; exact duration and field context are uncertain.'),
    (candidate, '2026-09-01', 27,
     'contextual', 'Routing/ownership clarification and forwarding, not timed master-data entry.'),
    (candidate, '2026-09-01', 28,
     'contextual', 'Routing/ownership clarification and forwarding, not timed master-data entry.'),
    (candidate, '2026-09-08', 25,
     'mixed_or_uncertain', 'Exact is updated after a requested-component case, but the updated object is unspecified.'),
    (candidate, '2026-09-23', 54,
     'contextual', 'Seven-minute blank-note clarification follows an item request; its precise work is unknown.'),
]

### Join every judgement back to the original row

The lists above store source keys and review judgements. The original activity, case, clock times and note are joined from the five CSVs one-to-one within each opportunity, with a check that each selected source row exists. The scope filter is then applied **before every comparison**, leaving removed memberships available only for the Appendix B audit.

Direct, mixed/uncertain and contextual evidence remain separate. Their associated minutes are whole observed intervals, not expected savings.


In [17]:
ledger = pd.DataFrame(evidence_rows, columns=[
    'opportunity', 'date', 'source_row', 'evidence_scope', 'inclusion_reason'
])
assert not ledger.duplicated(['opportunity', 'date', 'source_row']).any()

joined_parts = []
for candidate in opportunity_names:
    candidate_ledger = ledger.loc[ledger['opportunity'] == candidate]
    joined = candidate_ledger.merge(
        observations, on=['date', 'source_row'], how='left',
        validate='one_to_one', indicator=True
    )
    assert joined['_merge'].eq('both').all()
    joined_parts.append(joined.drop(columns='_merge'))
original_candidate_evidence = pd.concat(joined_parts, ignore_index=True)
removed_candidate_evidence = original_candidate_evidence.loc[~original_candidate_evidence['improvement_scope']].copy()
evidence = original_candidate_evidence.loc[original_candidate_evidence['improvement_scope']].copy()
ledger = evidence[['opportunity', 'date', 'source_row', 'evidence_scope', 'inclusion_reason']].copy()
assert evidence['improvement_scope'].all()
assert set(evidence['opportunity']) == set(opportunity_names)
assert removed_candidate_evidence['opportunity'].eq('O1').all()
assert len(removed_candidate_evidence) == 2
assert removed_candidate_evidence['minutes'].sum() == 8


## 4. Compare recorded burden, recurrence and evidence limits

Only eligible evidence for eligible candidates enters these tables. They keep observed burden separate from proposed feasibility. No 1–5 ratings, preference weights or gate verdicts are introduced. Names O1–O4 identify the eligible review proposals; their numbering is not a ranking.

**Associated minutes** sum the entire selected intervals. **Days seen** counts dates with any selected evidence, including untimed or contextual rows. **Case-days** uses the case label written in the CSV plus its date; it does not claim independent jobs or reassign mixed-case minutes. Notes and outcomes within one case are dependent observations.


In [18]:
opportunity_summary = evidence.groupby('opportunity').agg(
    selected_records=('source_row', 'size'),
    timed_records=('timed', 'sum'),
    untimed_records=('untimed', 'sum'),
    days_seen=('date', 'nunique'),
    case_days=('case_day', 'nunique')
)
opportunity_summary['associated_minutes'] = evidence.groupby('opportunity')['minutes'].sum(min_count=1)
opportunity_summary.insert(0, 'opportunity_name', opportunity_summary.index.map(opportunity_names))
display(opportunity_summary)

,opportunity_name,selected_records,timed_records,untimed_records,days_seen,case_days,associated_minutes
opportunity,,,,,,,
O1,Comparison evidence,11,11,0,4,10,62.0
O2,Order consolidation context,33,29,4,5,26,73.0
O3,Item/document retrieval,9,9,0,3,5,46.0
O4,Article/supplier information,17,13,4,4,6,43.0


### How much of that evidence is direct or mixed?

Read each row alongside its record and timed-record counts. A missing duration for an existing category means **not timed**, not zero. In particular, the two directly described MAX occurrences are untimed; their absence of timed minutes cannot be treated as no workload. Mixed and contextual minutes are not lower-quality savings estimates: no column estimates a saving at all.

In [19]:
scope_groups = evidence.groupby(['opportunity', 'evidence_scope'])
scope_summary = scope_groups.agg(
    records=('source_row', 'size'),
    timed_records=('timed', 'sum'),
    untimed_records=('untimed', 'sum'),
    days_seen=('date', 'nunique')
)
scope_summary['associated_minutes'] = scope_groups['minutes'].sum(min_count=1)
display(scope_summary)

records  ...  associated_minutes
opportunity evidence_scope               ...                    
O1          contextual                3  ...                 9.0
            direct                    6  ...                27.0
            mixed_or_uncertain        2  ...                26.0
O2          contextual               20  ...                45.0
            direct                    2  ...                 NaN
            mixed_or_uncertain       11  ...                28.0
O3          contextual                4  ...                14.0
            direct                    1  ...                 5.0
            mixed_or_uncertain        4  ...                27.0
O4          contextual                7  ...                17.0
            direct                    7  ...                16.0
            mixed_or_uncertain        3  ...                10.0

[12 rows x 5 columns]

### Dependence on a single date

The following check leaves the largest contributing day out of each opportunity total. It is a descriptive stress check, not a confidence interval. It asks whether the apparent burden depends mainly on one session; it does not estimate a normal week. A missing candidate/date total means there is no recorded timed total in the selected evidence for that date.

In [20]:
opportunity_days = evidence.groupby(['opportunity', 'date'])['minutes'].sum(min_count=1).unstack('date')
opportunity_days = opportunity_days.reindex(columns=dates)
display(opportunity_days)

day_dependence = pd.DataFrame(index=opportunity_summary.index)
day_dependence['associated_minutes'] = opportunity_summary['associated_minutes']
day_dependence['largest_day_minutes'] = opportunity_days.max(axis=1)
day_dependence['largest_day_share_pct'] = day_dependence['largest_day_minutes'] / day_dependence['associated_minutes'] * 100
day_dependence['minutes_without_largest_day'] = day_dependence['associated_minutes'] - day_dependence['largest_day_minutes']
display(day_dependence.round(1))

date,2026-08-31,2026-09-01,2026-09-08,2026-09-18,2026-09-23
opportunity,,,,,
O1,4.0,6.0,3.0,49.0,NaN
O2,10.0,15.0,10.0,14.0,24.0
O3,NaN,16.0,NaN,19.0,11.0
O4,17.0,11.0,8.0,NaN,7.0


,associated_minutes,largest_day_minutes,largest_day_share_pct,minutes_without_largest_day
opportunity,,,,
O1,62.0,49.0,79.0,13.0
O2,73.0,24.0,32.9,49.0
O3,46.0,19.0,41.3,27.0
O4,43.0,17.0,39.5,26.0


### Exclude contextual rows as a boundary check

This narrower view retains direct and mixed/uncertain evidence only. It does not split a mixed interval or prove the remaining time belongs wholly to the proposed function. Compare it with the full associated totals to see how much the chosen boundary matters.

In [21]:
direct_or_mixed = evidence.loc[evidence['evidence_scope'] != 'contextual']
boundary_check = direct_or_mixed.groupby('opportunity').agg(
    records=('source_row', 'size'),
    days_seen=('date', 'nunique'),
    timed_records=('timed', 'sum')
)
boundary_check['associated_minutes_without_context'] = direct_or_mixed.groupby('opportunity')['minutes'].sum(min_count=1)
boundary_check['all_selected_associated_minutes'] = opportunity_summary['associated_minutes']
display(boundary_check)

,records,days_seen,timed_records,associated_minutes_without_context,all_selected_associated_minutes
opportunity,,,,,
O1,8,4,8,53.0,62.0
O2,13,4,11,28.0,73.0
O3,5,3,5,32.0,46.0
O4,10,4,7,26.0,43.0


**What the comparison supports.** O1 now has **62 associated minutes** across four dates: **27 direct, 26 mixed/uncertain and 9 contextual**. Removing the EXC screenshot and partial-confirmation follow-up records removes eight minutes. September 18 still contributes 49 minutes (79.0%); only 13 remain when that date is omitted. All 91 CHECK minutes must not be labelled price checking.

O2 has **73 associated minutes** across all five dates: 28 mixed/uncertain and 45 contextual minutes. Its two direct MAX occurrences are untimed. Its largest day contributes 24 minutes, leaving 49 on other dates. The recurrence supports a closer look, but no interval isolates search from entry and ordering judgement.

O3 retains **46 minutes** (5 direct, 27 mixed/uncertain, 14 contextual), with 27 remaining after omitting its largest date. O4 retains **43 minutes** (16 direct, 10 mixed/uncertain, 17 contextual), with 26 remaining after its largest date. Their source notes describe retrieval, drawing and article/supplier preparation work; none of their selected rows is EXC or OTHER/EXC, and none describes the excluded aftercare or received-item search target. Their smaller totals do not establish lower value or easier implementation.

O5's former 53-minute total depended on **43 scope-excluded minutes**. Its residual ten minutes still concern delivery/payment follow-up; no distinct in-scope function is established. O5 is therefore removed entirely. O6 is also removed entirely: its five contextual CLAR minutes cannot make the excluded 42-minute received-item search an eligible target. Neither appears in the comparison, day-dependence tables or recommendation.


### Keep broad checking context and eligible unselected workload visible

All original CHECK records remain as checking context; all are eligible. The stage split is **37 Ordering minutes and 54 Monitoring minutes**, totalling **91**. The blank-note post-order clarification changes stage meaning without identifying the field checked. O1 draws only selected CHECK notes plus eligible CLAR/OTHER context; its total and the overall CHECK total measure different boundaries.


In [22]:
check_rows = observations.loc[observations['Activity'] == 'CHECK']
check_context = check_rows.groupby('date').agg(records=('source_row', 'size'))
check_context['recorded_minutes'] = check_rows.groupby('date')['minutes'].sum(min_count=1)
display(check_context)
print('All CHECK records:', len(check_rows))
print('All CHECK minutes:', check_rows['minutes'].sum(min_count=1))
check_by_stage = check_rows.groupby('stage').agg(records=('source_row', 'size'))
check_by_stage['recorded_minutes'] = check_rows.groupby('stage')['minutes'].sum(min_count=1)
assert check_rows['improvement_scope'].all()
assert check_by_stage['recorded_minutes'].to_dict() == {'Monitoring': 54.0, 'Ordering': 37.0}
display(check_by_stage)


,records,recorded_minutes
date,,
2026-08-31,5,13.0
2026-09-01,5,13.0
2026-09-08,1,3.0
2026-09-18,8,57.0
2026-09-23,2,5.0


All CHECK records: 21
All CHECK minutes: 91.0


,records,recorded_minutes
stage,,
Monitoring,15,54.0
Ordering,6,37.0


In [23]:
# Count overlapping memberships before measuring unique selected records.
memberships = ledger.groupby(['date', 'source_row'])['opportunity'].size()
repeated_keys = memberships.loc[memberships > 1].reset_index(name='opportunities_supported')
overlapping_evidence = evidence.merge(repeated_keys, on=['date', 'source_row'], how='inner', validate='many_to_one')
display(overlapping_evidence[['opportunity', 'date', 'source_row', 'minutes', 'evidence_scope']])

selected_keys = ledger[['date', 'source_row']].drop_duplicates()
selected_keys['in_review'] = True
coverage = scoped_observations.merge(selected_keys, on=['date', 'source_row'], how='left', validate='one_to_one')
selected_rows = coverage.loc[coverage['in_review'].eq(True)]
unselected_rows = coverage.loc[coverage['in_review'].isna()]
print('Eligible unique selected records:', len(selected_rows))
print('Unique selected recorded minutes:', selected_rows['minutes'].sum(min_count=1))
print('Eligible records outside the proposed-opportunity ledger:', len(unselected_rows))
print('Eligible recorded minutes outside that ledger:', unselected_rows['minutes'].sum(min_count=1))
assert len(ledger) == len(selected_rows) == 70
assert selected_rows['minutes'].sum() == 224
assert len(unselected_rows) == 176 and unselected_rows['minutes'].sum() == 345
assert len(selected_rows) + len(unselected_rows) == len(scoped_observations)
assert selected_rows['minutes'].sum() + unselected_rows['minutes'].sum() == 569
print('Share of eligible minutes selected (%):', round(selected_rows['minutes'].sum() / 569 * 100, 1))


,opportunity,date,source_row,minutes,evidence_scope


Eligible unique selected records: 70
Unique selected recorded minutes: 224.0
Eligible records outside the proposed-opportunity ledger: 176
Eligible recorded minutes outside that ledger: 345.0
Share of eligible minutes selected (%): 39.4


In [24]:
unselected_context = unselected_rows.groupby('Activity').agg(
    records=('source_row', 'size'),
    timed_records=('timed', 'sum'),
    untimed_records=('untimed', 'sum')
)
unselected_context['recorded_minutes'] = unselected_rows.groupby('Activity')['minutes'].sum(min_count=1)
display(unselected_context.sort_values('recorded_minutes', ascending=False))

,records,timed_records,untimed_records,recorded_minutes
Activity,,,,
CLAR,28,27,1,95.0
PO,34,34,0,94.0
SEND,40,34,6,68.0
OTHER,14,13,1,49.0
CHECK,13,13,0,38.0
[uncertain],1,1,0,1.0
DEC,1,0,1,NaN
REQ,8,0,8,NaN
REQ-Exact,8,0,8,NaN


**Coverage limit.** The eligible ledger contains **70 memberships over 70 distinct source rows**, comprising **224 of 569 eligible minutes (39.4%)**. The other **176 eligible records contain 345 minutes**. There is no current overlap between candidate memberships. The previously shared EXC row, 1 September source row 32, is excluded. Candidate totals describe separate review boundaries and should not be interpreted as additive savings.

The 41 scope-excluded records / 182 minutes are separate from the eligible unselected workload. Ordinary eligible records associated with excluded O5/O6 targets remain in the eligible unselected record; they do not reinstate those candidates. Eligible unselected work can contain other useful signals, including certificate handling, quotation lookup and request clarification. Appendix C preserves selected signals without turning excluded aftercare or physical searches into new candidates.


## 5. Defensible next steps before selecting a focal case

**Start with a bounded O1 source-document feasibility check.** Inspect paired price/date evidence from the six direct records (27 associated minutes across three dates) and confirm which field the buyer actually compares. This is a provisional investigation priority because these notes identify a concrete comparison action and reference sources. It is not a finding that O1 has the greatest benefit: 49 of its broader 62 minutes come from one date, and 26 minutes remain mixed or field-uncertain. The EXC screenshot and follow-up work are outside the proposed boundary.

**Keep O2 as the recurrent alternative.** Its evidence spans all five dates, but the 73 minutes comprise 28 mixed and 45 contextual minutes; direct MAX occurrences are untimed. Clarify whether an information-retrieval function can be separated from adding lines and deciding to order or hold. Do not interpret all consolidation context as AI-addressable search. O3 and O4 remain eligible alternatives if their documents, fields and review criteria can be established more clearly.

**Remove O5 and O6 from focal-case consideration.** The former order/issue-status priority relied mainly on excluded aftercare. Its residual non-EXC rows do not establish a separately bounded in-scope target. Received-item location is explicitly outside scope; its five contextual minutes do not rescue that candidate. Appendix B retains the audit without recommending either for improvement work.

The meeting and observer correction establish the exclusion boundary. The five CSVs still do **not** establish access permission, current data availability, trustworthy reference answers, human acceptance, implementation effort or demonstrated business value. No eligible candidate has passed a feasibility gate on this evidence.


### Questions that could change the provisional starting point

| Question | What to check and why it matters |
|---|---|
| Is the eligible function correctly bounded? | For O1, identify the compared field and the buyer's discrepancy decision. For O2, separate search, entry and order/hold judgement. Keep excluded aftercare and physical-location work outside both boundaries; do not allocate mixed minutes retrospectively. |
| Are suitable inputs accessible and permitted? | Inspect O1's paired documents, values and versions. For O2–O4, inspect the specific order/demand, item-history, drawing or article/supplier sources required. |
| What counts as a correct answer? | Establish an accountable owner and trustworthy reference for each comparison, match or field. Historical entries and later outcomes are not automatically correct reference answers. |
| Would the people doing the work accept the aid? | Verify who reviews uncertain results, corrects them, acts on them and maintains source data. Obtain permission and acceptance for the actual eligible function. |
| Can benefit and quality be tested fairly? | Agree comparable examples, manual and assisted procedures, end-to-end effort and error/omission checks. Measure effort transferred to colleagues as well as buyer time. |

**Conditional selection rule.** Continue with O1 only if its comparison field, paired sources and a useful buyer-reviewed output can be defined within scope and evaluated. Otherwise compare O2–O4 using these same questions. Agree decision criteria with the decision-maker before declaring a winner; the observations do not validate a final focal case or establish that AI is required.


## Appendix A. Traceable eligible opportunity evidence

Only eligible candidate memberships appear here. Each table shows the original CSV note in full. The record label includes date, source row, original case and original activity. `NaN` in minutes means not timed. Blank notes remain blank; their limited meaning is explained by the inclusion reason.

The inclusion reasons are analyst interpretations for this fresh review, not extra observations. Mixed-case work remains on its original source row. No missing note, referenced case note or unstated field is inferred as fact. The observer clarifications near the beginning are kept separately from these original notes.


### O1. Comparison evidence

In [25]:
candidate_evidence = evidence.loc[evidence['opportunity'] == 'O1'].copy()
candidate_evidence = candidate_evidence.sort_values(['date', 'source_row'])
candidate_evidence['record'] = (
    candidate_evidence['date'] + ' row ' + candidate_evidence['source_row'].astype(str)
    + ' | ' + candidate_evidence['Case'] + ' | ' + candidate_evidence['Activity']
)
notes_view = candidate_evidence.set_index('record')[['evidence_scope', 'minutes', 'Note']]
display(notes_view)
reasons_view = candidate_evidence.set_index('record')[['inclusion_reason']]
display(reasons_view)

,evidence_scope,minutes,Note
record,,,
2026-08-31 row 45 | OBS-12 | CHECK,direct,4.0,Geen leverdatum
2026-09-01 row 30 | OBS-18 | CHECK,mixed_or_uncertain,6.0,Pre-control: delivery time adjusted; price control could not be completed because of an unpaid-invoice issue. Supplier identity omitted.
2026-09-08 row 21 | OBS-08 | CHECK,direct,3.0,"Price comparison; check first the physical quotation, then the website."
2026-09-18 row 3 | OBS-02 | CLAR,contextual,5.0,"Price mismatch; Kayne requested information, not a purchase."
2026-09-18 row 7 | OBS-01 | OTHER,contextual,2.0,Attach the file to Exact.
2026-09-18 row 23 | OBS-02 | OTHER,contextual,2.0,Attach the order confirmation to the PO.
2026-09-18 row 36 | OBS-03 | CHECK,direct,7.0,Post-order price check.
2026-09-18 row 37 | OBS-09 | CHECK,direct,5.0,Post-order price check.
2026-09-18 row 38 | OBS-11 | CHECK,direct,2.0,Post-order price check.


,inclusion_reason
record,
2026-08-31 row 45 | OBS-12 | CHECK,The note identifies a missing delivery date.
2026-09-01 row 30 | OBS-18 | CHECK,Delivery-time adjustment and a blocked price check share one interval; no split is supported.
2026-09-08 row 21 | OBS-08 | CHECK,The note explicitly describes price comparison using quotation and website.
2026-09-18 row 3 | OBS-02 | CLAR,"Information request about a price mismatch, not an explicitly timed comparison."
2026-09-18 row 7 | OBS-01 | OTHER,"Attachment/filing steps are related document handling, not measured comparison time."
2026-09-18 row 23 | OBS-02 | OTHER,"Attachment/filing steps are related document handling, not measured comparison time."
2026-09-18 row 36 | OBS-03 | CHECK,The note explicitly names a post-order price check.
2026-09-18 row 37 | OBS-09 | CHECK,The note explicitly names a post-order price check.
2026-09-18 row 38 | OBS-11 | CHECK,The note explicitly names a post-order price check.


### O2. Order consolidation context

In [26]:
candidate_evidence = evidence.loc[evidence['opportunity'] == 'O2'].copy()
candidate_evidence = candidate_evidence.sort_values(['date', 'source_row'])
candidate_evidence['record'] = (
    candidate_evidence['date'] + ' row ' + candidate_evidence['source_row'].astype(str)
    + ' | ' + candidate_evidence['Case'] + ' | ' + candidate_evidence['Activity']
)
notes_view = candidate_evidence.set_index('record')[['evidence_scope', 'minutes', 'Note']]
display(notes_view)
reasons_view = candidate_evidence.set_index('record')[['inclusion_reason']]
display(reasons_view)

,evidence_scope,minutes,Note
record,,,
2026-08-31 row 8 | OBS-04 | REQ-mail,contextual,NaN,New article to old PO
2026-08-31 row 10 | OBS-04 | PO,contextual,2.0,1L added
2026-08-31 row 37 | OBS-11 | DEC,direct,NaN,MAX attempt failed; initial outcome was HOLD / leave until tomorrow.
2026-08-31 row 38 | OBS-11 | PO,mixed_or_uncertain,3.0,MAX/PO episode. Initial decision was to leave it until tomorrow because the MAX attempt failed. About one minute later he returned and ordered it anyway; reason is unknown.
2026-08-31 row 39 | OBS-11 | PO,contextual,NaN,Returned approximately one minute later and ordered after the earlier HOLD intention. Exact resumption time and reason were not captured; retained as an untimed known resumption.
2026-08-31 row 51 | OBS-15 | DEC,direct,NaN,MAX
2026-08-31 row 52 | OBS-15 | PO,mixed_or_uncertain,5.0,"MAX and PO occurred seamlessly: 12L original; 4L added; 16L total. Treat the 5 minutes as combined PO/MAX active time, not pure decision time."
2026-09-01 row 45 | OBS-23 | PO,contextual,4.0,HOLD outcome observed. MAX check was not explicitly captured: `MAXOBS=?`.
2026-09-01 row 46 | OBS-24 | PO,mixed_or_uncertain,2.0,"`MAXOBS=Y; MAX=?; HOLD`. MAX occurred, but ADD/NONE result was not separately captured."


,inclusion_reason
record,
2026-08-31 row 8 | OBS-04 | REQ-mail,Request to add an article to an old PO and its added line are explicit; a MAX/same-supplier search is not recorded.
2026-08-31 row 10 | OBS-04 | PO,Request to add an article to an old PO and its added line are explicit; a MAX/same-supplier search is not recorded.
2026-08-31 row 37 | OBS-11 | DEC,Untimed DEC notes explicitly identify MAX; occurrence only.
2026-08-31 row 38 | OBS-11 | PO,MAX and PO work share the recorded interval; the note explicitly rejects a pure decision-time interpretation.
2026-08-31 row 39 | OBS-11 | PO,Untimed return after HOLD intention; not a measured additional consolidation interval.
2026-08-31 row 51 | OBS-15 | DEC,Untimed DEC notes explicitly identify MAX; occurrence only.
2026-08-31 row 52 | OBS-15 | PO,MAX and PO work share the recorded interval; the note explicitly rejects a pure decision-time interpretation.
2026-09-01 row 45 | OBS-23 | PO,HOLD is observed while MAX is explicitly unknown.
2026-09-01 row 46 | OBS-24 | PO,"MAX is explicitly observed inside a PO episode; search, addition and outcome are not separately timed."


### O3. Item/document retrieval

In [27]:
candidate_evidence = evidence.loc[evidence['opportunity'] == 'O3'].copy()
candidate_evidence = candidate_evidence.sort_values(['date', 'source_row'])
candidate_evidence['record'] = (
    candidate_evidence['date'] + ' row ' + candidate_evidence['source_row'].astype(str)
    + ' | ' + candidate_evidence['Case'] + ' | ' + candidate_evidence['Activity']
)
notes_view = candidate_evidence.set_index('record')[['evidence_scope', 'minutes', 'Note']]
display(notes_view)
reasons_view = candidate_evidence.set_index('record')[['inclusion_reason']]
display(reasons_view)

,evidence_scope,minutes,Note
record,,,
2026-09-01 row 20 | OBS-12 | CLAR,mixed_or_uncertain,16.0,MI + historical search: old PO lookup; several matching items; required size/information not supplied.
2026-09-18 row 27 | OBS-10 | PO,contextual,7.0,Prepare drawings to attach to the PO; write down which drawings are missing.
2026-09-18 row 28 | OBS-10 | CLAR,mixed_or_uncertain,2.0,Look for the relevant drawings and ask the engineer for them.
2026-09-18 row 31 | OBS-11 | CLAR,mixed_or_uncertain,4.0,Relevant person unavailable; ask other colleagues. Also discuss the OBS-10 drawing.
2026-09-18 row 40 | OBS-10 | SEND,contextual,2.0,Drawing received from the engineer.
2026-09-18 row 41 | OBS-10 | CLAR,contextual,4.0,Final check that the drawing and its information were complete.
2026-09-23 row 34 | OBS-12 | CLAR,mixed_or_uncertain,5.0,"Look for the relevant drawing; unable to find it, ask a colleague."
2026-09-23 row 38 | OBS-14 | CLAR,direct,5.0,Look into the history of tubes following Emiel's request to order a tube.
2026-09-23 row 40 | OBS-14 | CLAR,contextual,1.0,Ask Emiel for confirmation; see the tube specification note below.


,inclusion_reason
record,
2026-09-01 row 20 | OBS-12 | CLAR,"Old-PO lookup, matching alternatives and missing required information share one interval."
2026-09-18 row 27 | OBS-10 | PO,"Preparing drawings and recording missing ones is associated document work, not explicitly retrieval alone."
2026-09-18 row 28 | OBS-10 | CLAR,Drawing search and asking the engineer share one interval.
2026-09-18 row 31 | OBS-11 | CLAR,Colleague enquiries for OBS-11 and an OBS-10 drawing discussion share four minutes across two cases; leave unallocated.
2026-09-18 row 40 | OBS-10 | SEND,Drawing receipt is not measured search time.
2026-09-18 row 41 | OBS-10 | CLAR,Drawing completeness checking is a distinct judgement.
2026-09-23 row 34 | OBS-12 | CLAR,Unsuccessful drawing search and asking a colleague share one interval.
2026-09-23 row 38 | OBS-14 | CLAR,Item history is explicitly searched; object type beyond tube history is unspecified.
2026-09-23 row 40 | OBS-14 | CLAR,Requester confirmation follows the item-history investigation; separate referenced case note unavailable.


### O4. Article/supplier information

In [28]:
candidate_evidence = evidence.loc[evidence['opportunity'] == 'O4'].copy()
candidate_evidence = candidate_evidence.sort_values(['date', 'source_row'])
candidate_evidence['record'] = (
    candidate_evidence['date'] + ' row ' + candidate_evidence['source_row'].astype(str)
    + ' | ' + candidate_evidence['Case'] + ' | ' + candidate_evidence['Activity']
)
notes_view = candidate_evidence.set_index('record')[['evidence_scope', 'minutes', 'Note']]
display(notes_view)
reasons_view = candidate_evidence.set_index('record')[['inclusion_reason']]
display(reasons_view)

,evidence_scope,minutes,Note
record,,,
2026-08-31 row 19 | OBS-07 | REQ-email,direct,NaN,Request: article input in Exact
2026-08-31 row 20 | OBS-07 | CLAR,contextual,2.0,Write up information
2026-08-31 row 21 | OBS-07 | OTHER,direct,4.0,Input to Exact
2026-08-31 row 22 | OBS-07 | SEND,contextual,NaN,Asking price? Instantaneous/tally occurrence; exact duration not measured because it happens in 30 seonds
2026-08-31 row 23 | OBS-07 | OTHER,direct,2.0,Attached info to article
2026-08-31 row 24 | OBS-07 | OTHER,direct,2.0,Attached info to article
2026-08-31 row 25 | OBS-07 | CLAR,mixed_or_uncertain,2.0,2nd item that needs be inputted; asking question
2026-08-31 row 26 | OBS-07 | CLAR,contextual,4.0,"2nd item, different comp."
2026-08-31 row 63 | OBS-07 | CLAR,contextual,1.0,Clarification needed for item 3; waiting for response.


,inclusion_reason
record,
2026-08-31 row 19 | OBS-07 | REQ-email,Untimed request explicitly asks for article input in Exact.
2026-08-31 row 20 | OBS-07 | CLAR,"Same article-input case contains information preparation and clarification, but fields and action split are not recorded."
2026-08-31 row 21 | OBS-07 | OTHER,Input to Exact follows the explicit article-input request in the same case.
2026-08-31 row 22 | OBS-07 | SEND,Untimed price question in the article-input case; exact duration and field context are uncertain.
2026-08-31 row 23 | OBS-07 | OTHER,Supporting information is attached to an article.
2026-08-31 row 24 | OBS-07 | OTHER,Supporting information is attached to an article.
2026-08-31 row 25 | OBS-07 | CLAR,Second item needs input; asking a question is the observed action.
2026-08-31 row 26 | OBS-07 | CLAR,"Same article-input case contains information preparation and clarification, but fields and action split are not recorded."
2026-08-31 row 63 | OBS-07 | CLAR,"Same article-input case contains information preparation and clarification, but fields and action split are not recorded."


## Appendix B. Excluded records and removed candidate memberships

This is an audit only. The first table retains every excluded raw row, including its original note, raw stage and activity. The reporting label and analytical stage are separate columns. EXC and OTHER/EXC contribute no improvement-comparison minutes.

The following membership audit explains the eight minutes removed from O1 and the complete removal of O5/O6. O5's former 53 minutes include 43 scope-excluded minutes and ten non-EXC minutes. O6's former 47 minutes include the excluded 42-minute physical search and five contextual CLAR minutes. Their non-EXC rows remain part of the 569-minute eligible record but outside every eligible candidate.


In [29]:
excluded_audit = excluded_rows.sort_values(['date', 'source_row']).copy()
excluded_audit['record'] = (
    excluded_audit['date'] + ' row ' + excluded_audit['source_row'].astype(str)
    + ' | ' + excluded_audit['Case'] + ' | ' + excluded_audit['Activity']
)
display(excluded_audit.set_index('record')[
    ['minutes', 'reporting_activity', 'Van Weele stage', 'stage', 'Note']
])


,minutes,reporting_activity,Van Weele stage,stage,Note
record,,,,,
2026-08-31 row 14 | OBS-05 | EXC,NaN,OTHER,Monitoring,Monitoring,Sub-minute exception/aftercare action immediately after the CHECK. Retained as a tally occurrence; duration was not measured precisely and is excluded from timed active-minute totals.
2026-08-31 row 27 | OBS-04 | EXC,2.0,OTHER,,Monitoring,aftercare
2026-08-31 row 57 | OBS-17 | EXC,2.0,OTHER,,Monitoring,Aftercare of a PO from last week
2026-09-01 row 1 | OBS-01 | EXC,12.0,OTHER,Follow-up and evaluation,Follow-up and evaluation,"Prior-week PO aftercare. Delivered components were incorrect; drawing check and supplier communication required. Buyer briefly walked to ask a same-case question at 10:41 and returned at 10:42; this remains same-case active EXC work, not an interruption."
2026-09-01 row 5 | OBS-03 | EXC,3.0,OTHER,Follow-up and evaluation,Follow-up and evaluation,Wrong order received; interruption recorded before continuation.
2026-09-01 row 6 | OBS-03 | EXC,2.0,OTHER,Follow-up and evaluation,Follow-up and evaluation,Continuation of wrong-order aftercare.
2026-09-01 row 14 | OBS-08 | EXC,2.0,OTHER,,Monitoring,Exception/aftercare episode; subtype not captured reliably.
2026-09-01 row 16 | OBS-09 | EXC,2.0,OTHER,Monitoring,Monitoring,Delivery-time issue linked to an unpaid invoice.
2026-09-01 row 18 | OBS-11 | EXC,3.0,OTHER,Monitoring,Monitoring,Delivery-status tracing for 10 pieces of a component.


In [30]:
retired_evidence_rows = []
# Date, source row, evidence scope, and reason for including the record.
candidate = 'O5'
retired_evidence_rows += [
    (candidate, '2026-08-31', 61,
     'direct', 'Supplier explicitly reports unpaid earlier POs blocking the current order.'),
    (candidate, '2026-08-31', 62,
     'contextual', 'Finance is informed after the payment blockage.'),
    (candidate, '2026-09-01', 16,
     'direct', 'Delivery-time issue is explicitly linked to unpaid invoice.'),
    (candidate, '2026-09-01', 18,
     'direct', 'Delivery status is explicitly traced.'),
    (candidate, '2026-09-01', 32,
     'mixed_or_uncertain', 'Partial confirmation and supplier follow-up share one interval; also relevant to comparison evidence.'),
    (candidate, '2026-09-08', 3,
     'contextual', 'Finance unpaid is explicit but the specific action is not.'),
    (candidate, '2026-09-08', 12,
     'mixed_or_uncertain', 'Supplier call asks when and why and forwards information; exact issue is not fully specified.'),
    (candidate, '2026-09-18', 5,
     'mixed_or_uncertain', 'Supplier contact concerns finance and a delivery-date change due to open invoice.'),
    (candidate, '2026-09-18', 8,
     'direct', 'Buyer asks whether several PO items can be delivered sooner; an explicit delivery follow-up.'),
    (candidate, '2026-09-18', 35,
     'contextual', 'Supplier assurance following an internal meeting is resolution context, not independent verification of a cause.'),
    (candidate, '2026-09-18', 44,
     'direct', 'Untimed request explicitly concerns PO status.'),
    (candidate, '2026-09-18', 45,
     'mixed_or_uncertain', 'PO review and forwarding information share an interval.'),
    (candidate, '2026-09-18', 46,
     'mixed_or_uncertain', 'Delivery follow-up and incorrect component size share an interval.'),
    (candidate, '2026-09-18', 71,
     'direct', 'Backorder and a date are recorded; exact action is sparse.'),
    (candidate, '2026-09-23', 27,
     'contextual', 'The note names an RC transport-order check, not a status check. Status relevance is inferred from the same-case expected-delivery forwarding at source row 30.'),
    (candidate, '2026-09-23', 30,
     'direct', 'Information on expected delivery is forwarded.'),
    (candidate, '2026-09-23', 44,
     'mixed_or_uncertain', 'ETA enquiry and a payment issue share an interval.'),
]
# Date, source row, evidence scope, and reason for including the record.
candidate = 'O6'
retired_evidence_rows += [
    (candidate, '2026-09-23', 51,
     'direct', 'The note explicitly records helping locate an item already received and stored by Logistics.'),
    (candidate, '2026-09-23', 49,
     'contextual', 'Same-case request and clarification precede the search, but their contents are not recorded.'),
    (candidate, '2026-09-23', 50,
     'contextual', 'Same-case request and clarification precede the search, but their contents are not recorded.'),
]
retired_ledger = pd.DataFrame(retired_evidence_rows, columns=[
    'opportunity', 'date', 'source_row', 'evidence_scope', 'inclusion_reason'
])
retired_evidence = retired_ledger.merge(
    observations, on=['date', 'source_row'], how='left', validate='one_to_one', indicator=True
)
assert retired_evidence['_merge'].eq('both').all()
retired_evidence = retired_evidence.drop(columns='_merge')
retired_evidence['candidate_exclusion_reason'] = retired_evidence['opportunity'].map({
    'O5': 'Aftercare/status-follow-up target removed; residual non-EXC records do not define a separate eligible function.',
    'O6': 'Received-item physical-location target explicitly outside scope; context cannot make it eligible.'
})
retired_summary = retired_evidence.groupby('opportunity').agg(records=('source_row', 'size'))
retired_summary['former_associated_minutes'] = retired_evidence.groupby('opportunity')['minutes'].sum(min_count=1)
retired_summary['scope_excluded_minutes'] = retired_evidence.loc[~retired_evidence['improvement_scope']].groupby('opportunity')['minutes'].sum(min_count=1)
retired_summary['non_EXC_minutes_still_removed_with_candidate'] = retired_evidence.loc[retired_evidence['improvement_scope']].groupby('opportunity')['minutes'].sum(min_count=1)
assert retired_summary['former_associated_minutes'].to_dict() == {'O5': 53.0, 'O6': 47.0}
assert retired_summary['scope_excluded_minutes'].to_dict() == {'O5': 43.0, 'O6': 42.0}
display(retired_summary)

removed_memberships = pd.concat([removed_candidate_evidence, retired_evidence], ignore_index=True)
removed_memberships['removal_reason'] = removed_memberships['scope_exclusion_reason']
retired_mask = removed_memberships['opportunity'].isin(['O5', 'O6'])
removed_memberships.loc[retired_mask, 'removal_reason'] = removed_memberships.loc[retired_mask, 'candidate_exclusion_reason']
removed_memberships = removed_memberships.sort_values(['opportunity', 'date', 'source_row'])
removed_memberships['record'] = (
    removed_memberships['date'] + ' row ' + removed_memberships['source_row'].astype(str)
    + ' | ' + removed_memberships['Case'] + ' | ' + removed_memberships['Activity']
)
display(removed_memberships.set_index(['opportunity', 'record'])[
    ['evidence_scope', 'minutes', 'improvement_scope', 'Note']
])
display(removed_memberships.set_index(['opportunity', 'record'])[
    ['inclusion_reason', 'removal_reason']
])


,records,former_associated_minutes,scope_excluded_minutes,non_EXC_minutes_still_removed_with_candidate
opportunity,,,,
O5,17,53.0,43.0,10.0
O6,3,47.0,42.0,5.0


evidence_scope  ...                                                                                                                                                      Note
opportunity record                                                     ...                                                                                                                                                          
O1          2026-09-01 row 32 | OBS-19 | EXC               contextual  ...                                                                               Supplier confirmation covered only part of the PO; follow-up call required.
            2026-09-18 row 74 | OBS-29 | EXC                   direct  ...                                                                       Make screenshots of the €410 price difference and save them for the following week.
O5          2026-08-31 row 61 | OBS-16 | CLAR                  direct  ...  Supplier call. There are old POs for which Finance has not paid. The supplier informed Arno that they therefore could not proceed with the OBS-16 order.
            2026-08-31 row 62 | OBS-16 | OTHER             contextual  ...                                                                                                                                            Inform Finance
            2026-09-01 row 16 | OBS-09 | EXC                   direct  ...                                                                                                          Delivery-time issue linked to an unpaid invoice.
            2026-09-01 row 18 | OBS-11 | EXC                   direct  ...                                                                                                     Delivery-status tracing for 10 pieces of a component.
            2026-09-01 row 32 | OBS-19 | EXC       mixed_or_uncertain  ...                                                                               Supplier confirmation covered only part of the PO; follow-up call required.
            2026-09-08 row 3 | OBS-02 | EXC                contextual  ...                                                                                                                                           Finance unpaid.
            2026-09-08 row 12 | OBS-05 | EXC       mixed_or_uncertain  ...                                                                                               Call supplier to ask when and why; forwarding to colleague.
            2026-09-18 row 5 | OBS-03 | OTHER/EXC  mixed_or_uncertain  ...                                    Supplier contact concerning finance and a delivery-date change due to an open invoice. Supplier recorded as Swagelock.
            2026-09-18 row 8 | OBS-01 | SEND                   direct  ...                                                                                              Ask whether several items on the PO can be delivered sooner.
            2026-09-18 row 35 | OBS-03 | EXC               contextual  ...                                                              Following an internal meeting, the supplier told Arno that the delay would not happen again.
            2026-09-18 row 44 | OBS-16 | REQ-mail              direct  ...                                                                                                                                    Request for PO status.
            2026-09-18 row 45 | OBS-16 | EXC       mixed_or_uncertain  ...                                                                        Arno checked the relevant POs and forwarded the information; three items involved.
            2026-09-18 row 46 | OBS-17 | EXC       mixed_or_uncertain  ...                                                                           Follow up on delivery and an incorrect component size; see clarification below.
            2026-09-18 row 71 | OBS-27 | EXC                   direct  ...                                                                                        

inclusion_reason                                                                                                   removal_reason
opportunity record                                                                                                                                                                                                                                                                                                                
O1          2026-09-01 row 32 | OBS-19 | EXC                                                             A partial supplier confirmation triggers follow-up; detecting and resolving it are not separately timed.                             EXC outside improvement scope; retained in the overall record under reporting OTHER.
            2026-09-18 row 74 | OBS-29 | EXC                                                                                                    EXC episode explicitly saves screenshots of the price difference.                             EXC outside improvement scope; retained in the overall record under reporting OTHER.
O5          2026-08-31 row 61 | OBS-16 | CLAR                                                                                          Supplier explicitly reports unpaid earlier POs blocking the current order.  Aftercare/status-follow-up target removed; residual non-EXC records do not define a separate eligible function.
            2026-08-31 row 62 | OBS-16 | OTHER                                                                                                                    Finance is informed after the payment blockage.  Aftercare/status-follow-up target removed; residual non-EXC records do not define a separate eligible function.
            2026-09-01 row 16 | OBS-09 | EXC                                                                                                          Delivery-time issue is explicitly linked to unpaid invoice.  Aftercare/status-follow-up target removed; residual non-EXC records do not define a separate eligible function.
            2026-09-01 row 18 | OBS-11 | EXC                                                                                                                                Delivery status is explicitly traced.  Aftercare/status-follow-up target removed; residual non-EXC records do not define a separate eligible function.
            2026-09-01 row 32 | OBS-19 | EXC                                                                Partial confirmation and supplier follow-up share one interval; also relevant to comparison evidence.  Aftercare/status-follow-up target removed; residual non-EXC records do not define a separate eligible function.
            2026-09-08 row 3 | OBS-02 | EXC                                                                                                            Finance unpaid is explicit but the specific action is not.  Aftercare/status-follow-up target removed; residual non-EXC records do not define a separate eligible function.
            2026-09-08 row 12 | OBS-05 | EXC                                                                        Supplier call asks when and why and forwards information; exact issue is not fully specified.  Aftercare/status-follow-up target removed; residual non-EXC records do not define a separate eligible function.
            2026-09-18 row 5 | OBS-03 | OTHER/EXC                                                                               Supplier contact concerns finance and a delivery-date change due to open invoice.  Aftercare/status-follow-up target removed; residual non-EXC records do not define a separate eligible function.
            2026-09-18 row 8 | OBS-01 | SEND                                                                         Buyer asks whether several PO items can be delivered sooner; an explicit delivery follow-up.  Aftercare/status-follow-up target removed; residual non-EXC records do not define a separate eligible func

## Appendix C. Other eligible raw-note signals and limits

These further signals come from the same five CSVs. Only eligible records appear here; any excluded source rows are already visible in Appendix B. They are not additional ranked candidates or measured benefits. Warehouse availability or collection coordination must remain distinct from the excluded search for an already received item.


In [31]:
other_signal_rows = [
    ('Certificate handling', '2026-09-18', 15),
    ('Certificate handling', '2026-09-18', 43),
    ('Certificate handling', '2026-09-18', 68),
    ('Certificate handling', '2026-09-18', 69),
    ('Incorrect deliveries / alternative sourcing', '2026-09-01', 1),
    ('Incorrect deliveries / alternative sourcing', '2026-09-01', 5),
    ('Incorrect deliveries / alternative sourcing', '2026-09-01', 6),
    ('Incorrect deliveries / alternative sourcing', '2026-09-01', 31),
    ('Incorrect deliveries / alternative sourcing', '2026-09-18', 46),
    ('Exact faults / communication friction', '2026-08-31', 5),
    ('Exact faults / communication friction', '2026-09-23', 21),
    ('Exact faults / communication friction', '2026-09-23', 41),
    ('Quotation lookup', '2026-09-18', 72),
    ('Quantity / visual request clarification', '2026-09-01', 3),
    ('Quantity / visual request clarification', '2026-09-01', 4),
    ('Warehouse availability / collection coordination', '2026-09-18', 19),
    ('Warehouse availability / collection coordination', '2026-09-18', 20),
    ('Warehouse availability / collection coordination', '2026-09-18', 21),
    ('Warehouse availability / collection coordination', '2026-09-18', 22),
]
other_signals = pd.DataFrame(other_signal_rows, columns=['signal', 'date', 'source_row'])
other_signals = other_signals.merge(observations, on=['date', 'source_row'], how='left', validate='one_to_one', indicator=True)
assert other_signals['_merge'].eq('both').all()
other_signals = other_signals.loc[other_signals['improvement_scope']].copy()
for signal in other_signals['signal'].drop_duplicates():
    print(signal)
    signal_rows = other_signals.loc[other_signals['signal'] == signal]
    display(signal_rows[['date', 'source_row', 'Activity', 'minutes', 'Note']])


Certificate handling


,date,source_row,Activity,minutes,Note
0,2026-09-18,15,OTHER,5.0,Interrupted to handle certification information; see case note below.


Exact faults / communication friction


,date,source_row,Activity,minutes,Note
9,2026-08-31,5,SEND,5.0,Error Exact
10,2026-09-23,21,OTHER,12.0,Fault in Exact; attempt to resolve it.
11,2026-09-23,41,SEND,1.0,Exact delay.


Quotation lookup


,date,source_row,Activity,minutes,Note
12,2026-09-18,72,CLAR,3.0,Look for the relevant quotation.


Quantity / visual request clarification


,date,source_row,Activity,minutes,Note
13,2026-09-01,3,CLAR,6.0,MI-requester: clarify whether requested quantity meant 18 total or 2 × 18.
14,2026-09-01,4,SEND,8.0,Request visual clarification from requester/colleague.


Warehouse availability / collection coordination


,date,source_row,Activity,minutes,Note
15,2026-09-18,19,CLAR,2.0,The item should be in the warehouse. Arno would ask Maurice and call back.
16,2026-09-18,20,CLAR,2.0,"Call with Maurice: three pieces available, whereas the note states eight were required."
17,2026-09-18,21,CLAR,1.0,Forward the information.
18,2026-09-18,22,CLAR,2.0,Tell service to prepare three pieces for collection; see case note below.


**Limits of the evidence.** These are five partial sessions with whole-minute recording, untimed occurrences, variable note detail and at least one episode already in progress. More detailed notes can produce more ledger entries without meaning that the underlying task is more important. Several notes refer to separate case notes that are not included in these CSVs; those references do not supply the missing information.

Records and case-days are not independent samples. A request, clarification, check and follow-up can describe one connected episode. Summed observed intervals are not complete-case lead times, avoidable effort, future workload or benefits. The raw files alone cannot establish task causes, current system capabilities, permissions or acceptance.

**Reproducibility.** Run the cells in order using the five original files dated 31 August, 1 September, 8 September, 18 September and 23 September 2026. The meeting scope is cited above; the dated stage clarifications and separate scope flag preserve the raw values. The eligible opportunity ledger and excluded-candidate audit are stored openly in the notebook as source-row judgements. No supplementary data file is required. Editing the CSVs or the ledger can change the tables and the interpretation; reconsider the narrative if either changes.
